[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vinod-seth/Applied-Scientist-Interview-Gauntlet/blob/main/tutorial/07_ml_from_scratch/ml_from_scratch_lab.ipynb)

# ML From Scratch Lab — Implementations and the Tests That Verify Them

**Session 7 armory notebook.** Every implementation in Lessons 1–4, written in NumPy and then **tested** — because the defining property of bugs in this round is that they do not crash. CPU-only, no model downloads, no network. The whole notebook runs in well under a minute.

▶ **[Open this notebook in Colab](https://colab.research.google.com/github/vinod-seth/Applied-Scientist-Interview-Gauntlet/blob/main/tutorial/07_ml_from_scratch/ml_from_scratch_lab.ipynb)** (text link, in case the badge above does not render in your viewer)

| Part | Lesson | Implementations | The tests that prove them |
|---|---|---|---|
| 1 | 1 — Attention | Softmax, scaled dot-product, multi-head, RoPE | Naive-loop reference, row sums, `H=1` identity, **leakage test**, RoPE norm and relative-position |
| 2 | 2 — Losses | `log_softmax`, cross-entropy, gradient, BCE, label smoothing | $\log K$ at init, overflow contrast at $z = 10^3$, **finite differences**, descent direction |
| 3 | 3 — Classical ML | k-means with k-means++, logistic regression | **Monotone inertia**, blob recovery, gradient check, separable-data divergence |
| 4 | 4 — Decoding | Temperature / top-*k* / top-*p*, beam search | Sampler identities, beam-1 = greedy, **brute-force enumeration**, length-penalty flip |

> **What to record:** nothing here belongs in your Metric Vault. Every number is a *mechanism demonstration* on synthetic data. What transfers to an interview is the **sentence** ("the mask goes on the scores as $-\infty$, and the leakage test proves it") and the **test**, never a figure from this notebook.

**Environment.** Pinned below; **last verified 2026-07-29** on Python 3.11 and on Colab's default runtime.

In [ ]:
%pip install -q "numpy>=1.26" "matplotlib>=3.8"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import itertools, warnings, sys

rng_global = np.random.default_rng(0)
np.set_printoptions(precision=4, suppress=True)
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

PASS, FAIL = "PASS", "FAIL"

def check(name, ok, detail=""):
    print(f"  [{PASS if ok else FAIL}] {name}" + (f"   {detail}" if detail else ""))
    assert ok, f"{name} failed. {detail}"

print("python", sys.version.split()[0], "| numpy", np.__version__)

---
## Part 1 — Attention, and four tests

Lesson 1. The reference implementation, then the tests from the blank-editor drill. **Attempt the drill before reading this.**

In [ ]:
def softmax(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)      # shift-invariant: exact, not approximate
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)


def sdpa(q, k, v, mask=None):
    """q,k,v: (..., T, d).  mask: True where FORBIDDEN.  -> output, weights"""
    d_k = q.shape[-1]
    scores = q @ np.swapaxes(k, -1, -2) / np.sqrt(d_k)      # NOT k.T on a 4-D array
    if mask is not None:
        scores = np.where(mask, -np.inf, scores)            # BEFORE the softmax
    w = softmax(scores, axis=-1)                            # over the KEY axis
    return w @ v, w


def split_heads(x, H):
    B, T, d_model = x.shape
    return x.reshape(B, T, H, d_model // H).transpose(0, 2, 1, 3)   # feature axis, then transpose


def merge_heads(x):
    B, H, T, dh = x.shape
    return x.transpose(0, 2, 1, 3).reshape(B, T, H * dh)            # transpose BEFORE reshape


def rope(x, positions, base=10000.0):
    """x: (..., T, d), d even. Rotates dimension pairs by position * theta_i."""
    d = x.shape[-1]
    i = np.arange(d // 2)
    theta = base ** (-2.0 * i / d)
    ang = positions[:, None] * theta[None, :]                       # (T, d/2)
    cos, sin = np.cos(ang), np.sin(ang)
    xe, xo = x[..., 0::2], x[..., 1::2]
    return np.stack([xe * cos - xo * sin, xe * sin + xo * cos], axis=-1).reshape(x.shape)


def multi_head_attention(x, Wq, Wk, Wv, Wo, H, causal=False, use_rope=False):
    B, T, d_model = x.shape
    q, k, v = split_heads(x @ Wq, H), split_heads(x @ Wk, H), split_heads(x @ Wv, H)
    if use_rope:
        pos = np.arange(T, dtype=float)
        q, k = rope(q, pos), rope(k, pos)                           # Q and K ONLY - never V
    mask = np.triu(np.ones((T, T), bool), k=1) if causal else None
    out, w = sdpa(q, k, v, mask)
    return merge_heads(out) @ Wo, w


B, T, d_model, H = 2, 6, 16, 4
r = np.random.default_rng(1)
x  = r.normal(size=(B, T, d_model))
Wq, Wk, Wv, Wo = (r.normal(size=(d_model, d_model)) / np.sqrt(d_model) for _ in range(4))
print("built:  x", x.shape, " d_model", d_model, " heads", H, " d_head", d_model // H)

In [ ]:
print("Part 1 tests")

# --- reference: the naive triple loop you actually trust ---------------------
def sdpa_naive(q, k, v):
    T, d = q.shape
    out = np.zeros_like(v)
    for i in range(T):
        s = np.array([q[i] @ k[j] / np.sqrt(d) for j in range(T)])
        s = np.exp(s - s.max()); s /= s.sum()
        out[i] = sum(s[j] * v[j] for j in range(T))
    return out

qq, kk, vv = r.normal(size=(5, 8)), r.normal(size=(5, 8)), r.normal(size=(5, 8))
fast, _ = sdpa(qq, kk, vv)
check("vectorised sdpa matches a naive triple loop", np.allclose(fast, sdpa_naive(qq, kk, vv)))

# --- test 1: output shape ---------------------------------------------------
out, w = multi_head_attention(x, Wq, Wk, Wv, Wo, H)
check("output shape is (B, T, d_model)", out.shape == (B, T, d_model), f"got {out.shape}")
check("score matrix is (B, H, T, T) - square in sequence length", w.shape == (B, H, T, T), f"got {w.shape}")

# --- test 2: attention rows sum to 1 (catches a wrong softmax axis) ---------
check("attention weights sum to 1 along the KEY axis", np.allclose(w.sum(-1), 1.0),
      f"max deviation {np.abs(w.sum(-1) - 1).max():.2e}")

# --- test 3: H=1 must equal single-head exactly (catches the missing transpose)
out1, _ = multi_head_attention(x, Wq, Wk, Wv, Wo, 1)
single, _ = sdpa(x @ Wq, x @ Wk, x @ Wv)
check("H=1 equals the single-head path", np.allclose(out1, single @ Wo))

# --- test 4: THE LEAKAGE TEST (catches a broken or misplaced causal mask) ---
x2 = x.copy(); x2[:, 3, :] += 5.0                       # perturb token 3 only
o_a, _ = multi_head_attention(x,  Wq, Wk, Wv, Wo, H, causal=True)
o_b, _ = multi_head_attention(x2, Wq, Wk, Wv, Wo, H, causal=True)
check("LEAKAGE: perturbing token 3 leaves outputs 0-2 bit-identical",
      np.array_equal(o_a[:, :3], o_b[:, :3]))
check("LEAKAGE control: output at position 3 DID change",
      not np.allclose(o_a[:, 3], o_b[:, 3]))

In [ ]:
print("Part 1 tests - RoPE")

y = r.normal(size=(7, 16))
pos = np.arange(7, dtype=float)
yr = rope(y, pos)
check("RoPE preserves vector norms (rotations are orthogonal)",
      np.allclose(np.linalg.norm(y, axis=-1), np.linalg.norm(yr, axis=-1)))

# relative position: score(3,5) must equal score(10,12) for identical content
qv, kv = r.normal(size=16), r.normal(size=16)
def rope_score(m, n):
    qm = rope(qv[None, :], np.array([float(m)]))[0]
    kn = rope(kv[None, :], np.array([float(n)]))[0]
    return qm @ kn
check("RoPE score depends only on relative position (n - m)",
      np.isclose(rope_score(3, 5), rope_score(10, 12)),
      f"{rope_score(3,5):.6f} vs {rope_score(10,12):.6f}")
check("...and DIFFERS at a different offset (control)",
      not np.isclose(rope_score(3, 5), rope_score(3, 9)))

In [ ]:
# Why the sqrt(d_k) divisor exists, measured.
ds = [4, 16, 64, 256, 1024]
raw_std, scaled_std, raw_maxp, scaled_maxp = [], [], [], []
rr = np.random.default_rng(3)
for d in ds:
    a, b = rr.normal(size=(4000, d)), rr.normal(size=(4000, d))
    s = np.einsum("id,id->i", a, b)
    raw_std.append(s.std()); scaled_std.append((s / np.sqrt(d)).std())
    row = np.concatenate([s[:15], np.zeros(1)])
    raw_maxp.append(softmax(row).max())
    scaled_maxp.append(softmax(row / np.sqrt(d)).max())

print(f"{'d_k':>6} | {'std(q.k)':>10} | {'sqrt(d_k)':>10} | {'std scaled':>11} | {'max softmax p':>14} | {'scaled':>8}")
print("-" * 74)
for d, a, b, c, e in zip(ds, raw_std, scaled_std, raw_maxp, scaled_maxp):
    print(f"{d:>6} | {a:>10.3f} | {np.sqrt(d):>10.3f} | {b:>11.3f} | {c:>14.4f} | {e:>8.4f}")

fig, ax = plt.subplots()
ax.plot(ds, raw_std, "o-", label="std of unscaled scores")
ax.plot(ds, np.sqrt(ds), "k--", lw=1, label=r"$\sqrt{d_k}$ (theory)")
ax.plot(ds, scaled_std, "s-", label=r"std after dividing by $\sqrt{d_k}$")
ax.set_xscale("log", base=2); ax.set_xlabel("$d_k$"); ax.set_ylabel("standard deviation of scores")
ax.set_title("Why the divisor is the standard deviation, not the dimension")
ax.legend(); plt.tight_layout(); plt.show()

**What you should have seen.** The unscaled score standard deviation tracks $\sqrt{d_k}$ exactly — that is the variance argument, measured rather than asserted — and the scaled line is flat at 1. The `max softmax p` columns show the consequence: without scaling, a single token takes almost all the probability mass as $d_k$ grows, which is the saturated softmax whose gradient vanishes.

**The leakage test is the one to remember.** Three lines, and it proves the property the causal mask exists for rather than proving the mask has the right shape.

---
## Part 2 — Losses, and the checks that catch silent bugs

Lesson 2. The stable forms, the $p - y$ gradient, and the four tests from the blank-editor drill.

In [ ]:
def log_softmax(z, axis=-1):
    m = np.max(z, axis=axis, keepdims=True)
    s = z - m
    return s - np.log(np.sum(np.exp(s), axis=axis, keepdims=True))


def softmax_cross_entropy(z, y, eps=0.0):
    """z: (N, K) logits. y: (N,) int labels. Returns (mean loss, grad wrt z)."""
    N, K = z.shape
    logp = log_softmax(z)
    target = np.full((N, K), eps / K)
    target[np.arange(N), y] += 1.0 - eps
    loss = -np.sum(target * logp) / N
    grad = (np.exp(logp) - target) / N          # p - y  (or p - y_smoothed)
    return loss, grad


def bce_with_logits(z, y, pos_weight=1.0):
    per = np.maximum(z, 0) - z * y + np.log1p(np.exp(-np.abs(z)))
    if pos_weight != 1.0:
        per = per * np.where(y == 1, pos_weight, 1.0)
    return float(np.mean(per))


def grad_check(f, z, analytic, h=1e-5):
    """Central differences in float64. Returns max relative error."""
    z = z.astype(np.float64).copy()
    num = np.zeros_like(z)
    it = np.nditer(z, flags=["multi_index"])
    while not it.finished:
        idx = it.multi_index
        old = z[idx]
        z[idx] = old + h; plus = f(z)
        z[idx] = old - h; minus = f(z)
        z[idx] = old
        num[idx] = (plus - minus) / (2 * h)
        it.iternext()
    return float(np.max(np.abs(num - analytic) /
                        np.maximum(1e-12, np.abs(num) + np.abs(analytic))))

print("defined: log_softmax, softmax_cross_entropy, bce_with_logits, grad_check")

In [ ]:
print("Part 2 tests")
rl = np.random.default_rng(5)

# --- test 1: log K at initialisation ----------------------------------------
for K in (2, 4, 10):
    z0 = np.zeros((32, K)); y0 = rl.integers(0, K, 32)
    loss0, _ = softmax_cross_entropy(z0, y0)
    check(f"K={K:>2}: loss at zero logits equals log K", np.isclose(loss0, np.log(K)),
          f"{loss0:.6f} vs log {K} = {np.log(K):.6f}")

# --- test 2: no overflow at extreme logits ----------------------------------
z_big = rl.normal(size=(8, 5)) * 1000.0
y_big = rl.integers(0, 5, 8)
stable_loss, _ = softmax_cross_entropy(z_big, y_big)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    naive = np.exp(z_big) / np.exp(z_big).sum(-1, keepdims=True)   # the naive path
    naive_loss = -np.mean(np.log(naive[np.arange(8), y_big]))
check("stable form is finite at |z| ~ 1e3", np.isfinite(stable_loss), f"loss = {stable_loss:.2f}")
check("naive softmax-then-log is NOT finite there", not np.isfinite(naive_loss),
      f"naive = {naive_loss}")

# --- test 3: finite-difference gradient check --------------------------------
zz = rl.normal(size=(6, 4)); yy = rl.integers(0, 4, 6)
for eps in (0.0, 0.1):
    _, g = softmax_cross_entropy(zz, yy, eps=eps)
    rel = grad_check(lambda t: softmax_cross_entropy(t, yy, eps=eps)[0], zz, g)
    check(f"gradient matches central differences (eps={eps})", rel < 1e-7, f"rel err {rel:.2e}")

# --- test 4: a step along -grad decreases the loss ---------------------------
l0, g = softmax_cross_entropy(zz, yy)
l1, _ = softmax_cross_entropy(zz - 1e-3 * g, yy)
check("a small step along -grad decreases the loss", l1 < l0, f"{l0:.6f} -> {l1:.6f}")

# --- BCE: the stable form vs the naive one -----------------------------------
zb = np.array([-800.0, -3.0, 0.0, 3.0, 800.0]); yb = np.array([0., 0., 1., 1., 1.])
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    sig = 1.0 / (1.0 + np.exp(-zb))
    naive_bce = float(np.mean(-(yb * np.log(sig) + (1 - yb) * np.log(1 - sig))))
check("stable BCE is finite at |z| = 800", np.isfinite(bce_with_logits(zb, yb)),
      f"{bce_with_logits(zb, yb):.4f}")
check("naive BCE is not", not np.isfinite(naive_bce), f"naive = {naive_bce}")

In [ ]:
# Label smoothing: what it does to the optimal logit gap.
K = 5
gaps = np.linspace(0, 14, 200)
fig, ax = plt.subplots()
for eps in (0.0, 0.05, 0.1, 0.2):
    curve = []
    for g in gaps:
        z = np.zeros((1, K)); z[0, 0] = g
        curve.append(softmax_cross_entropy(z, np.array([0]), eps=eps)[0])
    curve = np.array(curve)
    ax.plot(gaps, curve, label=f"eps = {eps}")
    if eps > 0:
        ax.plot(gaps[curve.argmin()], curve.min(), "ko", ms=5)

ax.set_xlabel("logit gap between the true class and the rest")
ax.set_ylabel("loss")
ax.set_title("Hard targets are minimised only at infinity; smoothing gives a finite optimum")
ax.legend(); plt.tight_layout(); plt.show()

for eps in (0.05, 0.1, 0.2):
    curve = np.array([softmax_cross_entropy(np.array([[g] + [0.0] * (K - 1)]),
                                            np.array([0]), eps=eps)[0] for g in gaps])
    print(f"eps = {eps:<5}  optimal logit gap = {gaps[curve.argmin()]:5.2f}")
print(f"eps = 0.0    optimal logit gap = infinity (the curve never turns)")

**What you should have seen.** With $\varepsilon = 0$ the loss decreases monotonically in the logit gap — the optimum is at infinity, so training pushes confidence up forever. Each smoothed curve turns and has a **finite minimum**, marked with a dot, and larger $\varepsilon$ moves that optimum closer to zero. That is the whole mechanism behind the calibration benefit.

And the two contrast tests above are the demonstration to reach for in an interview: *"the naive form returns `inf` at these logits and the stable one returns a number"* is more convincing than any explanation of why.

---
## Part 3 — k-means and logistic regression, with property-based tests

Lesson 3. Note test 1: the **monotone inertia assertion** comes free from the convergence proof and catches essentially every k-means bug.

In [ ]:
def kmeans_plusplus_init(X, k, rng):
    C = [X[rng.integers(len(X))]]
    for _ in range(1, k):
        d2 = np.min(((X[:, None, :] - np.array(C)[None, :, :]) ** 2).sum(-1), axis=1)
        tot = d2.sum()
        p = d2 / tot if tot > 0 else np.full(len(X), 1.0 / len(X))
        C.append(X[rng.choice(len(X), p=p)])
    return np.array(C)


def kmeans(X, k, iters=100, tol=1e-9, seed=0, init="++"):
    rng = np.random.default_rng(seed)
    C = kmeans_plusplus_init(X, k, rng) if init == "++" else X[rng.choice(len(X), k, replace=False)]
    history = []
    for _ in range(iters):
        d2 = ((X[:, None, :] - C[None, :, :]) ** 2).sum(-1)        # (N, k)
        labels = np.argmin(d2, axis=1)
        history.append(float(d2[np.arange(len(X)), labels].sum()))
        newC = np.empty_like(C)
        for j in range(k):
            m = labels == j
            # empty-cluster policy: reseed at the point furthest from its centroid
            newC[j] = X[m].mean(0) if m.any() else X[np.argmax(d2[np.arange(len(X)), labels])]
        if np.abs(newC - C).max() < tol:
            C = newC; break
        C = newC
    return C, labels, np.array(history)


def sigmoid(z):
    out = np.empty_like(z, dtype=float)
    pos = z >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
    e = np.exp(z[~pos]); out[~pos] = e / (1.0 + e)
    return out


def logistic_regression(X, y, lr=0.5, iters=800, l2=0.0):
    w = np.zeros(X.shape[1])
    losses = []
    for _ in range(iters):
        grad = X.T @ (sigmoid(X @ w) - y) / len(y) + l2 * w        # the same p - y
        w -= lr * grad
        losses.append(bce_with_logits(X @ w, y) + 0.5 * l2 * w @ w)
    return w, np.array(losses)

print("defined: kmeans_plusplus_init, kmeans, sigmoid, logistic_regression")

In [ ]:
print("Part 3 tests")
rk = np.random.default_rng(7)
centres = np.array([[0., 0.], [6., 6.], [0., 7.]])
X = np.vstack([c + rk.normal(scale=0.7, size=(150, 2)) for c in centres])

# --- test 1: THE property test - inertia can never increase ------------------
for seed in range(6):
    _, _, hist = kmeans(X, 3, seed=seed)
    check(f"seed {seed}: inertia monotonically non-increasing",
          bool(np.all(np.diff(hist) <= 1e-9)), f"{len(hist)} iterations, final {hist[-1]:.2f}")

# --- test 2: recovers three balanced blobs ----------------------------------
C, labels, hist = kmeans(X, 3, seed=0)
counts = np.sort(np.bincount(labels, minlength=3))
check("recovers 3 clusters of ~150 points each", bool(np.all(np.abs(counts - 150) <= 15)),
      f"counts {counts.tolist()}")

# --- k-means++ vs uniform seeding, over restarts ------------------------------
pp = [kmeans(X, 3, seed=s, init="++")[2][-1] for s in range(30)]
un = [kmeans(X, 3, seed=s, init="uniform")[2][-1] for s in range(30)]
print(f"\n  final inertia over 30 seeds:  k-means++  mean {np.mean(pp):7.2f}  worst {np.max(pp):7.2f}")
print(f"                                uniform    mean {np.mean(un):7.2f}  worst {np.max(un):7.2f}")

In [ ]:
print("Part 3 tests - logistic regression")
rg = np.random.default_rng(11)
n = 200
Xa = np.hstack([rg.normal(size=(n, 2)), np.ones((n, 1))])          # bias column
w_true = np.array([1.5, -2.0, 0.3])
ya = (rg.random(n) < sigmoid(Xa @ w_true)).astype(float)

# --- test 3: gradient check --------------------------------------------------
w0 = rg.normal(size=3) * 0.5
def nll(w):
    return bce_with_logits(Xa @ w, ya)
analytic = Xa.T @ (sigmoid(Xa @ w0) - ya) / n
rel = grad_check(nll, w0, analytic)
check("logistic gradient matches central differences", rel < 1e-7, f"rel err {rel:.2e}")

w_fit, losses = logistic_regression(Xa, ya, iters=2000)
check("training loss decreases monotonically", bool(np.all(np.diff(losses) <= 1e-12)),
      f"{losses[0]:.4f} -> {losses[-1]:.4f}")

# --- test 4: separable data - the optimum is at infinity ---------------------
Xs = np.hstack([np.vstack([rg.normal(loc=-4, scale=0.8, size=(60, 2)),
                           rg.normal(loc=+4, scale=0.8, size=(60, 2))]), np.ones((120, 1))])
ys = np.r_[np.zeros(60), np.ones(60)]
w_probe, _ = logistic_regression(Xs, ys, iters=2000, l2=0.0)
check("the two classes really are perfectly separable",
      bool(np.all((sigmoid(Xs @ w_probe) > 0.5) == (ys == 1))))

budgets = [1_000, 5_000, 25_000]
free = [np.linalg.norm(logistic_regression(Xs, ys, iters=n, l2=0.0)[0]) for n in budgets]
reg  = [np.linalg.norm(logistic_regression(Xs, ys, iters=n, l2=0.1)[0]) for n in budgets]

print(f"\n  {'iterations':>11} | {'||w|| no L2':>12} | {'||w|| L2=0.1':>13}")
print("  " + "-" * 42)
for n, a, b in zip(budgets, free, reg):
    print(f"  {n:>11,} | {a:>12.3f} | {b:>13.3f}")

check("without L2 the weight norm keeps growing with the iteration budget",
      free[2] > free[1] > free[0] and free[2] / free[1] > 1.05,
      f"{free[0]:.2f} -> {free[1]:.2f} -> {free[2]:.2f} (no finite optimum)")
check("with L2 the weight norm converges",
      abs(reg[2] - reg[1]) / reg[1] < 0.01,
      f"{reg[1]:.4f} -> {reg[2]:.4f} (converged)")

w_free = logistic_regression(Xs, ys, iters=25_000, l2=0.0)[0]
w_reg = logistic_regression(Xs, ys, iters=25_000, l2=0.1)[0]
print(f"\n  max predicted probability, unregularised: {sigmoid(Xs @ w_free).max():.8f}")
print(f"  max predicted probability, L2 = 0.1     : {sigmoid(Xs @ w_reg).max():.8f}")
print("  The unregularised fit reports near-certainty it has not earned - the")
print("  optimum is at infinity, so training never converges, it only slows down.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for s in range(6):
    _, _, h = kmeans(X, 3, seed=s)
    axes[0].plot(h, marker="o", ms=3, alpha=.8)
axes[0].set_xlabel("iteration"); axes[0].set_ylabel("inertia")
axes[0].set_title("Inertia never increases - the free unit test")

for j, m in enumerate(["o", "s", "^"]):
    axes[1].scatter(X[labels == j, 0], X[labels == j, 1], s=10, marker=m, alpha=.6)
axes[1].scatter(C[:, 0], C[:, 1], c="k", marker="X", s=160, label="centroids")
axes[1].set_title("Recovered clusters"); axes[1].legend()
plt.tight_layout(); plt.show()

**What you should have seen.** Every inertia curve descends and flattens — never a single upward step, across every seed. That is not luck; it is what Lesson 3's two-step proof guarantees, which is exactly why `assert np.all(np.diff(inertia) <= 0)` is such a strong test: a violation is a *guaranteed* defect, not a suspicious result.

The separable-data numbers are the other thing worth noting. Without $L_2$, the maximum predicted probability runs to essentially 1.0 and the weight norm keeps growing — a model reporting certainty it has not earned. With a penalty, both stay finite.

---
## Part 4 — Decoding, verified by identity

Lesson 4. Decoders are unusually testable, and Part 4's second test — **brute-force enumeration** — is the strongest verification anywhere in this session.

In [ ]:
def sample_next(logits, temperature=1.0, top_k=None, top_p=None, rng=None):
    rng = rng or np.random.default_rng()
    z = np.asarray(logits, dtype=float)
    if temperature <= 0:
        return int(np.argmax(z))                       # T -> 0 is exactly greedy
    z = z / temperature                                # 1. temperature FIRST
    if top_k is not None and top_k < len(z):           # 2. truncate in the logit domain
        kth = np.partition(z, -top_k)[-top_k]
        z = np.where(z < kth, -np.inf, z)
    if top_p is not None and top_p < 1.0:
        order = np.argsort(z)[::-1]
        cum = np.cumsum(softmax(z[order]))
        keep = order[:int(np.searchsorted(cum, top_p)) + 1]        # INCLUDE the crossing token
        masked = np.full_like(z, -np.inf); masked[keep] = z[keep]; z = masked
    p = softmax(z)                                     # 3. renormalises over survivors
    return int(rng.choice(len(p), p=p))                # 4. sample


def length_penalty(length, alpha=0.0):
    return 1.0 if alpha == 0.0 else ((5.0 + length) ** alpha) / (6.0 ** alpha)


def beam_search(step_fn, bos, eos, beam_width=3, max_len=6, alpha=0.0):
    live, finished = [([bos], 0.0)], []
    for _ in range(max_len):
        if not live:
            break
        cand = []
        for seq, sc in live:
            lp = step_fn(seq)
            for tok in np.argsort(lp)[::-1][:beam_width]:
                cand.append((seq + [int(tok)], sc + float(lp[tok])))
        cand.sort(key=lambda t: t[1], reverse=True)
        live = []
        for seq, sc in cand:
            if len(live) == beam_width:
                break
            if seq[-1] == eos:
                finished.append((seq, sc))            # retire it; never expand again
            else:
                live.append((seq, sc))                # only unfinished beams continue
    finished.extend(live)                             # hit max_len without EOS
    finished.sort(key=lambda t: t[1] / length_penalty(len(t[0]), alpha), reverse=True)
    return finished


def greedy_decode(step_fn, bos, eos, max_len=6):
    seq, score = [bos], 0.0
    for _ in range(max_len):
        lp = step_fn(seq)
        t = int(np.argmax(lp))
        seq.append(t); score += float(lp[t])
        if t == eos:
            break
    return seq, score

print("defined: sample_next, length_penalty, beam_search, greedy_decode")

In [ ]:
print("Part 4 tests - samplers")
rs = np.random.default_rng(13)
logits = np.array([3.0, 2.0, 1.5, 0.2, -1.0, -4.0])

check("temperature -> 0 equals argmax",
      sample_next(logits, temperature=0.0) == int(np.argmax(logits)))

# top_p = 1.0 and top_k = V must both reduce to untruncated sampling
def freq(**kw):
    r_ = np.random.default_rng(99)
    c = np.zeros(len(logits))
    for _ in range(40000):
        c[sample_next(logits, rng=r_, **kw)] += 1
    return c / 40000

base, full_p, full_k = freq(), freq(top_p=1.0), freq(top_k=len(logits))
check("top_p = 1.0 matches untruncated sampling", np.allclose(base, full_p, atol=0.01))
check("top_k = V matches untruncated sampling", np.allclose(base, full_k, atol=0.01))

# sampled frequencies match the truncated distribution within 3 standard errors
emp = freq(top_p=0.8)
z = logits.copy()
order = np.argsort(z)[::-1]; cum = np.cumsum(softmax(z[order]))
keep = order[:int(np.searchsorted(cum, 0.8)) + 1]
m = np.full_like(z, -np.inf); m[keep] = z[keep]
theory = softmax(m)
se = np.sqrt(np.maximum(theory * (1 - theory), 1e-12) / 40000)
worst = float(np.max(np.abs(emp - theory) / se))
check("top_p = 0.8 frequencies match theory within 3 SE", worst < 3.5, f"worst {worst:.2f} SE")
print(f"\n  nucleus at p=0.8 keeps {len(keep)} of {len(logits)} tokens: {sorted(keep.tolist())}")

# temperature changes WHICH tokens are in the nucleus - the order-of-operations point
for temp in (0.5, 1.0, 2.0):
    zt = logits / temp
    o = np.argsort(zt)[::-1]; c = np.cumsum(softmax(zt[o]))
    print(f"  temperature {temp:>4}: nucleus at p=0.8 holds {int(np.searchsorted(c, 0.8)) + 1} tokens")

In [ ]:
print("Part 4 tests - beam search")
V, MAXLEN, BOS = 4, 4, 0
rb = np.random.default_rng(10)          # a table where greedy is deliberately suboptimal
logT = np.log(softmax(rb.normal(size=(V, V)), axis=-1))     # log P(next | last)

def step_fn(seq):
    return logT[seq[-1]]

# --- test 1: beam width 1 equals greedy, exactly -----------------------------
b1 = beam_search(step_fn, BOS, eos=-1, beam_width=1, max_len=MAXLEN)[0]
g = greedy_decode(step_fn, BOS, eos=-1, max_len=MAXLEN)
check("beam width 1 equals greedy decoding exactly",
      b1[0] == g[0] and np.isclose(b1[1], g[1]), f"{b1[0]} vs {g[0]}")

# --- test 2: THE STRONG ONE - brute-force enumeration ------------------------
best_seq, best_score = None, -np.inf
for combo in itertools.product(range(V), repeat=MAXLEN):
    s, prev = 0.0, BOS
    for t in combo:
        s += logT[prev, t]; prev = t
    if s > best_score:
        best_score, best_seq = s, [BOS] + list(combo)
wide = beam_search(step_fn, BOS, eos=-1, beam_width=V ** MAXLEN, max_len=MAXLEN)[0]
check(f"a wide beam matches brute force over all {V ** MAXLEN} sequences",
      wide[0] == best_seq and np.isclose(wide[1], best_score),
      f"beam {wide[0]} ({wide[1]:.4f}) vs brute {best_seq} ({best_score:.4f})")

# ...and a narrow beam is NOT guaranteed to - beam search is a heuristic
narrow = beam_search(step_fn, BOS, eos=-1, beam_width=1, max_len=MAXLEN)[0]
print(f"\n  width 1  : {narrow[0]}  score {narrow[1]:.4f}")
print(f"  width 256: {wide[0]}  score {wide[1]:.4f}   <- the true argmax")
check("on this table greedy is SUBOPTIMAL - beam search is a heuristic, not exact",
      narrow[0] != wide[0] and narrow[1] < wide[1],
      f"greedy loses {wide[1] - narrow[1]:.4f} of log-probability")

# --- test 3: a finished beam is retired but retained -------------------------
EOS = 3
logT2 = logT.copy(); logT2[BOS] = np.log(softmax(np.array([0., 0., 0., 4.])))   # EOS very likely first
out = beam_search(lambda s: logT2[s[-1]], BOS, eos=EOS, beam_width=3, max_len=MAXLEN)
early = [s for s, _ in out if EOS in s]
check("a beam that emitted EOS is present in the output", len(early) > 0)
check("no tokens were generated after EOS",
      all(s.index(EOS) == len(s) - 1 for s in early),
      f"example {early[0]}")

# --- test 4: length normalisation flips the ranking --------------------------
# short has the better RAW score; long has the better score PER TOKEN
short = ([BOS, 1, 2], -2.0)
long_ = ([BOS, 1, 2, 1, 2, 1, 2, 1, 2, 1], -3.0)
print()
for a in (0.0, 0.7):
    for seq, sc in (short, long_):
        print(f"  alpha = {a}: length {len(seq):>2}  raw {sc:>5.1f}  "
              f"normalised {sc / length_penalty(len(seq), a):>7.4f}")
    ranked = sorted([short, long_], key=lambda t: t[1] / length_penalty(len(t[0]), a), reverse=True)
    print(f"    -> winner has length {len(ranked[0][0])}\n")
check("alpha = 0 prefers the short sequence",
      max([short, long_], key=lambda t: t[1] / length_penalty(len(t[0]), 0.0))[0] == short[0])
check("alpha = 0.7 prefers the long sequence",
      max([short, long_], key=lambda t: t[1] / length_penalty(len(t[0]), 0.7))[0] == long_[0])

In [ ]:
# How close does each beam width get to the true optimum?
widths = [1, 2, 3, 4, 8, 16, 64, 256]
scores = [beam_search(step_fn, BOS, eos=-1, beam_width=w, max_len=MAXLEN)[0][1] for w in widths]

fig, ax = plt.subplots()
ax.plot(widths, scores, "o-", label="best score found by beam search")
ax.axhline(best_score, color="#059669", ls="--", lw=2, label="true argmax (brute force)")
ax.set_xscale("log", base=2); ax.set_xlabel("beam width (log scale)")
ax.set_ylabel("total log-probability")
ax.set_title(f"Beam search is a heuristic: it reaches the optimum only when wide enough")
ax.legend(); plt.tight_layout(); plt.show()

first_exact = next(w for w, s in zip(widths, scores) if np.isclose(s, best_score))
print(f"smallest tested beam width that found the true argmax on this problem: {first_exact}")

**What you should have seen.** Beam width 1 reproduces greedy decoding exactly, and a sufficiently wide beam reproduces brute-force enumeration exactly. Between them, the curve shows the heuristic closing on the optimum — and the fact that a narrow beam *can* miss it is the concrete evidence behind "beam search is not guaranteed to find the most probable sequence."

The temperature-versus-nucleus table above is the other thing to carry: the same $p = 0.8$ admits a different number of tokens at each temperature, which is precisely why temperature must be applied **before** truncation.

---
## What to take from this notebook

| Part | The test you should be able to name in ten seconds |
|---|---|
| 1 | "Attention rows sum to one, and a leakage test proves causality — perturb token 3, assert outputs 0–2 are unchanged." |
| 1 | "With `n_heads=1` it must equal the single-head path, which catches the missing transpose." |
| 2 | "The loss at initialisation must be $\log K$ — that one line catches label misalignment, a double softmax, and a wrong class count." |
| 2 | "Central finite differences in float64, $h \approx 10^{-5}$, relative error below $10^{-7}$." |
| 3 | "`assert np.all(np.diff(inertia) <= 0)` — it comes free from the convergence proof and catches nearly every k-means bug." |
| 3 | "Logistic regression is convex, so no restarts — but on separable data the weights diverge without $L_2$." |
| 4 | "Beam width 1 equals greedy exactly, and on a toy vocabulary a wide beam must match brute-force enumeration." |

**Every test above is one or two lines and needs no reference implementation** — each asserts a *property implied by the objective*. That is what makes them usable on code you wrote sixty seconds ago, in front of an interviewer.

> **Reminder:** none of the numbers here are yours to quote. They are synthetic data on this machine. What transfers is the mechanism, the shape, and the test.

**Back to:** [Session 7 README](README.md) · [Mock Round](05_mock_round.md) · [Chapter Quiz](quiz.md)